# 05 - Expression Derivation and Constraint Solving

> **When to use**: When columns have computational relationships (e.g., `short_code = project_no[-6:]`), or have UNIQUE constraints.
>
> **Core concept**: `derive_from` declares column dependencies, `expression` specifies the formula, ColumnDAG auto-topological sort.

## Applicable Scenarios

- Column B depends on column A's value (e.g., abbreviation, substring, concatenation) → `derive_from` + `expression`
- Column has UNIQUE constraint → ConstraintSolver auto-backtracking
- Large UNIQUE workloads → understand bounded retries and memory costs
- Need chained dependencies (A → B → C) → ColumnDAG topological sort

## What You Will Learn

- `derive_from` + `expression` column derivation
- ColumnDAG topological sort
- ExpressionEngine 26 safe functions
- UNIQUE constraint backtracking
- Optional hash-based ConstraintSolver mode

See architecture.zh-CN.md §6

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| **→ 05** | **Expression Derivation and Constraint Solving** | **Core: DAG / Expression** | **01** |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---
## Setup

Use Python 3.10+ and run this notebook from `examples/notebooks` in a repository checkout. Select a notebook kernel from the environment containing these packages:

```bash
python -m pip install 'sqlseed[mimesis]==0.2.4' 'sqlseed-cli==0.2.4' jupyterlab
```

For source development, install Core and CLI together as described in the [repository README](../../README.md). This notebook creates its own temporary database and cache. Run cells from top to bottom; the validation helpers raise on partial generation or failed CLI commands.


In [ ]:
PRJ_PATTERN = "PRJ-\\d{6}"
# Install the packages listed in Setup into the selected notebook kernel.
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
import os
import tempfile
from pathlib import Path
notebook_temp = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-05-")
work_dir = Path(notebook_temp.name)
os.environ["SQLSEED_CACHE_DIR"] = str(work_dir / "cache")
db_path = build(work_dir / "demo.db")

# Fail visibly if a generation only partially succeeds.
generation_checks = []
def check_result(result, expected_count):
    if result.errors or result.count != expected_count:
        raise RuntimeError(f"{result.table_name}: expected {expected_count}, wrote {result.count}; errors={result.errors}")
    generation_checks.append({"table": result.table_name, "count": result.count, "errors": list(result.errors)})
    print(f"Verified {result.table_name}: {result.count} rows; errors={result.errors}")
    return result

def check_results(results, config_path):
    config = sqlseed.load_config(str(config_path))
    expected = {table.name: table.count for table in config.tables}
    for result in results:
        check_result(result, expected[result.table_name])
    if {result.table_name for result in results} != set(expected):
        raise RuntimeError("Not every configured table produced a result")
    return results

def check_cli(result):
    if result.exit_code != 0:
        raise RuntimeError(result.output) from result.exception
    return result


# Populate base dependencies
with connect(str(db_path)) as orch:
    check_result(orch.fill_table("organizations", count=5, seed=42), 5)
    check_result(orch.fill_table("members", count=20, seed=42), 20)
    check_result(orch.fill_table("projects", count=10, seed=42), 10)
    check_result(orch.fill_table("tags", count=8, seed=42), 8)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Data Stream Generation | `src/sqlseed/core/stream.py` | `DataStream` |
| Dependency Sorting | `src/sqlseed/core/column_dag.py` | `ColumnDAG` |
| Constraint Backtracking | `src/sqlseed/core/constraints.py`| `ConstraintSolver` |

> Corresponding architecture diagram: [§6 Column Dependency DAG and Constraint Backtracking](../../docs/architecture.zh-CN.md#6-列依赖-dag-与约束回溯)

## 1. See It in Action — The Power of Derived Columns

Often, columns have dependencies: `short_code` is the last 6 chars of `project_no`, `description` is concatenated from `project_no`.

sqlseed's `derive_from` + `expression` lets you declare such relationships, **auto-generating correct derived values**:

In [ ]:

with connect(str(db_path)) as orch:
    preview = orch.preview_table('projects', count=5, columns={
        'project_no': {'generator': 'pattern', 'params': {'pattern': PRJ_PATTERN}},
        'short_code': {'derive_from': 'project_no', 'expression': 'value[-6:]'},
        'description': {'derive_from': 'project_no', 'expression': "'Project-' + value"},
    })
    print(f"{'project_no':<15s}  {'short_code':<10s}  {'description':<30s}")
    print('-' * 60)
    for row in preview:
        print(f"{row['project_no']:<15s}  {row['short_code']:<10s}  {row['description']:<30s}")

Just declare the dependency, sqlseed auto:

1. **Topological sort** — determines generation order (first `project_no`, then `short_code` and `description`)
2. **Expression evaluation** — uses `simpleeval` safe engine to execute expressions
3. **Timeout protection** — complex expressions have a 5s timeout; timed-out threads cannot be killed

Below we break down each mechanism in detail.

## 2. derive_from + expression

When a column's value depends on another, use `derive_from` to declare the dependency, `expression` to specify the formula:

```yaml
columns:
  - name: project_no
    generator: pattern
    params:
      regex: "PRJ-\\d{6}"
  - name: short_code
    derive_from: project_no
    expression: "value[-6:]"
```

sqlseed first generates `project_no`, then computes `short_code` using `value[-6:]`.

In [ ]:
import sqlite3

result = check_result(fill(
    str(db_path),
    table="projects",
    count=5, clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "value[-6:]"},
    },
), 5)


conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}")
conn.close()

## 3. ColumnDAG Topological Sort

When multi-level dependencies exist, `ColumnDAG` auto-performs topological sort to ensure correct generation order:

```
project_no → short_code (value[-6:])
           → description (concat('Project ', value))
```

If cyclic dependencies exist, sqlseed raises `CyclicDependencyError`.

In [ ]:
result = check_result(fill(
    str(db_path),
    table="projects",
    count=5,
    clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "value[-6:]"},
        "description": {"derive_from": "project_no", "expression": "concat('Project ', value)"},
    },
), 5)

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, description FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}, description={row[2]}")
conn.close()

## 4. ExpressionEngine 26 Safe Functions

The expression engine uses `simpleeval` with these actual built-ins:

| Category | Functions |
|---|---|
| Conversion | `len`, `int`, `str`, `float`, `hex`, `oct`, `bin` |
| Numeric | `abs`, `min`, `max`, `round` |
| String | `upper`, `lower`, `strip`, `lstrip`, `rstrip`, `zfill`, `replace`, `substr`, `lpad`, `rpad`, `concat` |
| Random | `random_float`, `random_int`, `random_choice` |
| Date/time | `timedelta` |

Python slicing and arithmetic are also supported. `lookup` is added only when an adapter is supplied. Functions such as `substring`, `ceil`, `clamp`, `md5`, and `sha256` are not in SAFE_FUNCTIONS.

In [ ]:
result = check_result(fill(
    str(db_path),
    table="projects",
    count=3,
    clear_before=True,
    columns={
        "project_no": {"type": "pattern", "regex": PRJ_PATTERN},
        "short_code": {"derive_from": "project_no", "expression": "upper(value[-6:])"},
        "name": {"derive_from": "project_no", "expression": "concat('Project-', value)"},
    },
), 3)

conn = sqlite3.connect(str(db_path))
rows = conn.execute("SELECT project_no, short_code, name FROM projects").fetchall()
for row in rows:
    print(f"project_no={row[0]}, short_code={row[1]}, name={row[2]}")
conn.close()

## 5. ExpressionTimeoutError

Complex expressions execute in a worker thread with a default 5-second timeout. Timeout raises `ExpressionTimeoutError`; the underlying thread cannot be killed. Simple expressions run directly. Treat this as bounded waiting, not isolation for arbitrary Python execution.

## 6. UNIQUE Constraint Backtracking

When a column has a UNIQUE constraint, `ConstraintSolver` uses a backtracking algorithm to ensure no duplicates:

1. Generate a value
2. Check if it conflicts with existing values
3. If conflict, regenerate (retry up to N times)
4. If all retries fail, backtrack to the relevant source column; exhaustion reports an error

UNIQUE columns in the demo database:
- `members.member_no` (UNIQUE)
- `members.email` (UNIQUE)
- `projects.project_no` (UNIQUE)
- `tags.name` (UNIQUE)

In [ ]:
result = check_result(fill(str(db_path), table="members", count=50, clear_before=True), 50)

conn = sqlite3.connect(str(db_path))
member_nos = [r[0] for r in conn.execute("SELECT member_no FROM members").fetchall()]
emails = [r[0] for r in conn.execute("SELECT email FROM members").fetchall()]
print(f"Total rows: {len(member_nos)}")
print(f"Unique member_nos: {len(set(member_nos))}")
print(f"Unique emails: {len(set(emails))}")
print(f"All member_nos unique: {len(member_nos) == len(set(member_nos))}")
print(f"All emails unique: {len(emails) == len(set(emails))}")
conn.close()

## 7. Optional Hash-Based Constraint Tracking

`ConstraintSolver(probabilistic=True)` stores truncated SHA-256 hashes for single-column uniqueness tracking. It can report a false collision and still needs retries. Normal `fill` does not automatically enable this mode above a row threshold. Composite uniqueness retains exact tuple tracking. Database UNIQUE constraints remain authoritative.

## 🔗 Multi-level derive_from Chained Dependencies

`derive_from` supports chained dependencies: A → B → C, ColumnDAG auto-performs topological sort.

In [ ]:
with sqlseed.connect(str(db_path)) as orch:
    preview = orch.preview_table('projects', count=3, columns={
        'project_no': {'generator': 'pattern', 'params': {'pattern': PRJ_PATTERN}},
        'short_code': {'derive_from': 'project_no', 'expression': 'value[-6:]'},
        'description': {'derive_from': 'short_code', 'expression': "'Project-' + value"},
    })
    print('Multi-level derive_from chain (project_no -> short_code -> description):')
    for row in preview:
        print(f"  {row['project_no']} -> {row['short_code']} -> {row['description']}")

## ⚠️ In Practice: ExpressionTimeoutError

Expression execution exceeding 5s triggers timeout protection.

In [ ]:
from sqlseed.core.expression import ExpressionEngine

engine = ExpressionEngine(timeout_seconds=1)  # 1 second for demo

# Safe expressions execute instantly
result = engine.evaluate('abs(-42)', {})
print(f'abs(-42) = {result}')

# Demonstrate timeout API
print(f'\nExpressionEngine timeout={engine._timeout}s')
print('  - Expressions exceeding the timeout raise ExpressionTimeoutError')
print('  - Default timeout: 5 seconds')
print('  - Used internally by derive_from expressions')
print('  - Thread-based: cannot be killed, only detected')


In [ ]:
# A UNIQUE rule cannot override declared string-length or CHECK limits.
# Infeasible finite domains fail explicitly; inspect errors instead of assuming expansion.
result = check_result(fill(str(db_path), table="members", count=50, clear_before=True, seed=42), 50)
with sqlite3.connect(str(db_path)) as connection:
    total, unique_members, unique_emails = connection.execute(
        "SELECT count(*), count(DISTINCT member_no), count(DISTINCT email) FROM members"
    ).fetchone()
    assert (total, unique_members, unique_emails) == (50, 50, 50)
    print("Actual UNIQUE counts:", total, unique_members, unique_emails)


## 🔐 Composite Unique Constraints

`ConstraintSolver` supports multi-column composite unique constraints, ensuring combined values are not duplicated.

In [ ]:
from sqlseed.core.constraints import ConstraintSolver

solver = ConstraintSolver()

# Register composite unique (org_code, member_no)
pairs = set()
for i in range(20):
    org = f'ORG-{i % 3}'
    member = f'M{i:04d}'
    ok = solver.check_and_register_composite('org_member', (org, member))
    assert ok
    pairs.add((org, member))

print(f'Registered {len(pairs)} unique (org_code, member_no) pairs')
print(f'Sample: {list(pairs)[:3]}')
assert not solver.check_and_register_composite("org_member", ("ORG-0", "M0000"))


## Summary

| Feature | Description |
|------|------|
| derive_from | Declare column dependencies |
| expression | Compute derived values (based on simpleeval) |
| ColumnDAG | Topological sort to determine generation order |
| ExpressionEngine | 26 safe functions, 5s timeout protection |
| ConstraintSolver | UNIQUE backtracking + probabilistic mode |
| UniqueAdjuster | Adjust within schema and configured bounds; reject infeasible domains |
| Composite Unique | Multi-column composite unique |

**Next**: [06-config-deep-dive.ipynb](06-config-deep-dive.ipynb) — Config Model Deep Dive

In [ ]:
# Verify exact database totals after the full notebook, plus every declared FK.
import sqlite3
expected_counts = {'organizations': 5, 'members': 50, 'projects': 3, 'tasks': 0, 'tags': 8, 'reviews': 0}
with sqlite3.connect(str(db_path)) as verification_db:
    actual_counts = {
        table: verification_db.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0]
        for table in expected_counts  # Fixed tutorial table names.
    }
    assert actual_counts == expected_counts, (actual_counts, expected_counts)
    fk_errors = verification_db.execute("PRAGMA foreign_key_check").fetchall()
    assert not fk_errors, fk_errors
print("Verified database row counts:", actual_counts)
print("Database FK check:", fk_errors)
print("Verified fill operations:", len(generation_checks))
with sqlite3.connect(str(db_path)) as verification_db:
    mismatches = verification_db.execute(
        "SELECT COUNT(*) FROM projects WHERE short_code != substr(project_no, -6) "
        "OR name != 'Project-' || project_no"
    ).fetchone()[0]
    assert mismatches == 0
    print("Derived-column mismatches:", mismatches)
